# 07 — Vektordatabaser med ChromaDB

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 06

**Hva du bygger:** En vedvarende, søkbar kunnskapsbase med pensjonsdokumenter — lagret lokalt med ChromaDB, helt gratis.

---

## Hvorfor ikke bare bruke en liste?

I notatbok 06 beregnet vi cosinus-likhet mot *alle* dokumenter for hvert søk.
Med 1000 dokumenter er det OK. Med 1 million? Det tar minutter.

En **vektordatabase** løser dette med *approximate nearest neighbor* (ANN) — en smart indeks som finner de nærmeste vektorene på millisekunder, selv med millioner av dokumenter.

| Vektordatabase | Pris | Kjøremåte | Passer for |
|---------------|------|-----------|----------|
| **ChromaDB** | Gratis | Lokalt | Prototyper, læring |
| Qdrant | Gratis (lokalt) | Lokalt/sky | Produksjon |
| pgvector | Gratis | Postgres | Allerede bruker Postgres |
| Pinecone | Betalt | Sky | Stor skala |

Vi bruker **ChromaDB** — én pip-install, ingen Docker, ingen konto.

In [ ]:
%pip install -q chromadb sentence-transformers

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# Lokal embedding-modell (gratis)
embed_modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ChromaDB lagrer data på disk — vedvarer mellom kjøringer
klient = chromadb.PersistentClient(path="./chroma_db")

# En 'collection' er som en tabell i en vanlig database
samling = klient.get_or_create_collection(
    name="pensjonsdokumenter",
    metadata={"hnsw:space": "cosine"},  # Bruk cosinus-likhet
)

print(f"Samling klar: {samling.name}")
print(f"Antall dokumenter: {samling.count()}")

---

## Del 1: Legge til dokumenter

In [ ]:
# Pensjonsdokumenter med metadata
dokumenter = [
    {"id": "spk-001", "tekst": "AFP (avtalefestet pensjon) gir rett til tidligpensjon fra 62 år for ansatte i offentlig sektor.", "kilde": "spk", "kategori": "AFP"},
    {"id": "spk-002", "tekst": "Alderspensjon utbetales livsvarig fra 67 år. Beløpet avhenger av opptjeningstid og lønn.", "kilde": "spk", "kategori": "alderspensjon"},
    {"id": "spk-003", "tekst": "Uførepensjon innvilges ved varig nedsatt arbeidsevne på minst 20 prosent.", "kilde": "spk", "kategori": "uførepensjon"},
    {"id": "spk-004", "tekst": "Barnepensjon utbetales til barn under 20 år dersom en forsørger er død.", "kilde": "spk", "kategori": "barnepensjon"},
    {"id": "spk-005", "tekst": "Etterlattepensjon gir økonomisk støtte til ektefelle eller partner etter dødsfall.", "kilde": "spk", "kategori": "etterlattepensjon"},
    {"id": "spk-006", "tekst": "Pensjonsopptjening skjer ved å betale pensjonsinnskudd via arbeidsgiver over tid.", "kilde": "spk", "kategori": "opptjening"},
    {"id": "spk-007", "tekst": "Du kan søke om AFP tre måneder før ønsket uttaksdato via arbeidsgiver eller SPK.", "kilde": "spk", "kategori": "AFP"},
    {"id": "nav-001", "tekst": "NAV forvalter alderspensjon fra folketrygden, som er separat fra SPK-pensjonen.", "kilde": "nav", "kategori": "alderspensjon"},
]

# Embed alle tekster
tekster    = [d["tekst"] for d in dokumenter]
vektorer   = embed_modell.encode(tekster).tolist()
ids        = [d["id"] for d in dokumenter]
metadata   = [{"kilde": d["kilde"], "kategori": d["kategori"]} for d in dokumenter]

# Legg til i ChromaDB
samling.add(
    documents=tekster,
    embeddings=vektorer,
    ids=ids,
    metadatas=metadata,
)

print(f"Lagt til {len(dokumenter)} dokumenter. Totalt: {samling.count()}")

---

## Del 2: Søke i databasen

In [ ]:
def søk(spørsmål: str, topp_k: int = 3, filter: dict = None) -> None:
    spørsmål_vektor = embed_modell.encode([spørsmål]).tolist()
    
    kwargs = dict(query_embeddings=spørsmål_vektor, n_results=topp_k)
    if filter:
        kwargs["where"] = filter
    
    resultater = samling.query(**kwargs)
    
    print(f"Spørsmål: {spørsmål}")
    for i, (dok, dist, meta) in enumerate(zip(
        resultater["documents"][0],
        resultater["distances"][0],
        resultater["metadatas"][0],
    )):
        likhet = 1 - dist  # ChromaDB returnerer avstand, ikke likhet
        print(f"  [{likhet:.3f}] ({meta['kategori']}) {dok}")
    print()

søk("Kan jeg gå av tidlig med pensjon?")
søk("Hva skjer med familien min om jeg dør?")

In [ ]:
# Filtrert søk — bare dokumenter fra SPK, ikke NAV
print("Filtrert til kun SPK-dokumenter:")
søk("Alderspensjon", topp_k=2, filter={"kilde": "spk"})

---

## Del 3: Oppdatere og slette dokumenter

In [ ]:
# Oppdater et dokument
ny_tekst = "AFP gir rett til tidligpensjon fra 62 år. Fra 2025 er satsen justert."
ny_vektor = embed_modell.encode([ny_tekst]).tolist()

samling.update(
    ids=["spk-001"],
    documents=[ny_tekst],
    embeddings=ny_vektor,
)
print("Dokument spk-001 oppdatert.")

# Slett et dokument
# samling.delete(ids=["nav-001"])

# Hent et spesifikt dokument
resultat = samling.get(ids=["spk-001"])
print(f"Hentet: {resultat['documents'][0]}")

---

## Del 4: Forstå HNSW-indeksen

ChromaDB bruker **HNSW** (Hierarchical Navigable Small World) — en grafbasert indeks der nærliggende vektorer er koblet til hverandre.

```
Naivt søk:   Sammenlign med ALLE N vektorer → O(N)
HNSW-søk:    Naviger grafen → O(log N)  — 1000x raskere på 1M dokumenter
```

Du trenger ikke forstå detaljene — men det er greit å vite at vektordatabaser *ikke* gjør en komplett gjennomgang for hvert søk.

---

## Oppsummering

| Operasjon | ChromaDB-metode |
|-----------|----------------|
| Opprett samling | `get_or_create_collection(...)` |
| Legg til dokumenter | `samling.add(documents, embeddings, ids, metadatas)` |
| Semantisk søk | `samling.query(query_embeddings, n_results)` |
| Filtrert søk | `samling.query(..., where={"kilde": "spk"})` |
| Oppdater | `samling.update(ids, documents, embeddings)` |
| Slett | `samling.delete(ids)` |

---

## Hva er neste steg?

**Neste: `08_rag_basics.ipynb`** — Du kombinerer alt du har lært: embeddings + vektordatabase + LLM. Resultatet er et fungerende RAG-system som svarer på spørsmål basert på dine egne dokumenter.